In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import dask.array as da
import utils
import skimage as ski
import pandas as pd
# import time
# import extract_features
# import ast
# from cellpose.models import CellposeModel
# from cellpose.io import logger_setup
# logger_setup()
import dask_image.ndfilters as dask_ndi
from deeptile.extensions import stitch
import deeptile
import importlib
importlib.reload(utils)
import brieflow_segment_utils
importlib.reload(brieflow_segment_utils)
from brieflow_segment_utils import image_log_scale, reconcile_nuclei_cells, dask_image_log_scale, tiled_reconcile_nuclei_cells
from deeptile.core.lift import lift
from collections import defaultdict
from cellpose.models import CellposeModel

In [2]:
img_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18.ome.tif"
coverslip_path = "/Users/hannahbolen/Desktop/image_analysis/o8p_day18_coverslip.ome.tiff"

nuclei_mask_path = "/Users/hannahbolen/Desktop/image_analysis/here/o8p_day18_nuclei_premask.tif"
cyto_mask_path = "/Users/hannahbolen/Desktop/image_analysis/here/o8p_day18_cytoplasm_premask.tif"


nuclei_save_path = "/Users/hannahbolen/Desktop/image_analysis/here/o8p_day18_nuclei_mask.tif"
cyto_save_path = "/Users/hannahbolen/Desktop/image_analysis/here/o8p_day18_cytoplasm_mask.tif"

img_dtype = tifffile.TiffFile(img_path).pages[0].dtype

# coverslip = da.from_array(tifffile.imread(coverslip_path))
# img = da.from_zarr(tifffile.imread(img_path, aszarr=True))[0]
nuclei_mask = tifffile.memmap(nuclei_mask_path, mode='r')
cyto_mask = tifffile.memmap(cyto_mask_path, mode='r')
# cy5_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_cy5.tif"
# gfp_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_gfp.tif"

# gfp_path_pp = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_gfp_preprocess.tif"
# cy5_path_pp = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_cy5_preprocess.tif"

# cy5 = tifffile.memmap(cy5_path, mode='r')
# gfp = tifffile.memmap(gfp_path, mode='r')

In [ ]:
cy5log = image_log_scale(cy5)
cy5log = da.from_array(cy5log)
del cy5

In [ ]:
# preprocessing
img0 = img.copy()
# img1 = img[1].copy()

# log transform and normalize GFP channel
img0 = da.log(da.where(img0 == 0, 0.01, img0))
x01 = da.percentile(img0, 0.1, axis=(0, 1)).compute()
x99 = da.percentile(img0, 99, axis=(0, 1)).compute()
img0 = (img0 - x01) / (x99 - x01)
img0 = dask_ndi.gaussian_filter(img0, sigma=2)
img0 = img0*coverslip

In [ ]:
tile_size = (2048, 2048)
overlap = (0.1, 0.1)

nuclei_dt = deeptile.load(img0)

nuclei_tiles = nuclei_dt.get_tiles(tile_size, overlap)

In [ ]:
# # preprocessing
# img = np.log(np.where(img == 0, 0.01, img))
# x01 = np.percentile(img.compute(), .1)
# x99 = np.percentile(img.compute(), 99.99)
# img = (img - x01) / (x99 - x01)
# img=dask_ndi.gaussian_filter(img, sigma=2)
# img = img*coverslip


model_parameters = {'gpu': True}
nuclei_eval_parameters = {'diameter': 60, 'cellprob_threshold': 0.25, 'flow_threshold':.3, 'batch_size':16, 'min_size':400, 'normalize':False}
# model = CellposeModel(**model_parameters)
# mask, _, _ = model.eval(ti, **nuclei_eval_parameters)

cellpose = utils.cellpose_segmentation(model_parameters, nuclei_eval_parameters, threshold=0)
masks_nuclei = cellpose(nuclei_tiles)
mask_nuclei = stitch.stitch_masks(masks_nuclei)

tifffile.imwrite(nuclei_mask_path, mask_nuclei.astype(img_dtype), dtype=img_dtype)

In [ ]:
gfplog = image_log_scale(gfp)

In [ ]:
img = da.stack([img0, cy5log])
# img0 = dask_ndi.gaussian_filter(img0, sigma=2)
# img0 = img0*coverslip
img = dask_ndi.gaussian_filter(img, sigma=(0,2,2))
img = img*coverslip

In [ ]:
cy5log = image_log_scale(cy5)
gfplog = image_log_scale(gfp)

cy5 = cy5log
gfp = gfplog

del cy5log, gfplog

In [ ]:
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_gfp_preprocess.tif", gfp.astype(img_dtype), dtype=img_dtype, contiguous=True)
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day18_cy5_preprocess.tif", cy5.astype(img_dtype), dtype=img_dtype, contiguous=True)

In [ ]:
img = dask_ndi.gaussian_filter(img, sigma=(0,2,2))
img = img*coverslip

In [ ]:
model_parameters = {'gpu': True}
cyto_eval_parameters = {'diameter':100, 'batch_size':16, 'normalize':False}
# model = CellposeModel(**model_parameters)
# cyto_mask, _, _ = model.eval(full, **eval_parameters)
cellpose = utils.cellpose_segmentation(model_parameters, cyto_eval_parameters, threshold=0)
masks_cyto = cellpose(img_tiles)
mask_cyto = stitch.stitch_masks(masks_cyto)

tifffile.imwrite(cyto_mask_path, mask_cyto.astype(img_dtype), dtype=img_dtype)
# # fig, ax = plt.subplots(figsize=(20,20))
# # ax.imshow(ski.segmentation.mark_boundaries(img_log[1], cyto_mask, color=(1,0,1)))
# # ax.set_yticks([])
# # ax.set_xticks([])
# # plt.tight_layout()

In [ ]:
# img1 = img[1].copy()
# log transform and normalize cy5 channel
img = dask_image_log_scale(img[1])

In [ ]:
# log transform cy5 w different method
img1 =  dask_image_log_scale(img1)

# stack and gaussian blur
img = da.stack([img0, img1])
img = dask_ndi.gaussian_filter(img, sigma=(0,2,2))
img = img*coverslip

In [ ]:
plt.imshow(img_tiles[14,15][1])

In [ ]:
from cellpose.models import CellposeModel

In [ ]:
nuclei_mask_path

In [ ]:
tifffile.imwrite(nuclei_mask_path, mask_nuclei.astype(img_dtype), dtype=img_dtype)

In [ ]:
nuc_outline = ski.segmentation.find_boundaries(mask,mode="outer")
nuc_outline = np.ma.masked_where(nuc_outline==False,nuc_outline)

fig,ax = plt.subplots(figsize=(20,20))
ax.imshow(ski.exposure.rescale_intensity(ti, in_range=(0,3000)))
#show = microshow(images=composite, cmaps=['pure_green','pure_magenta'], limits=[[0,4000],[0,3500]], ax=ax)
ax.imshow(nuc_outline, cmap="Grays_r")

ax.set_yticks([])
ax.set_xticks([])
plt.tight_layout()

In [ ]:
cyto_tiles = nuclei_tiles.import_data(img, "image").unpad()

In [ ]:
nuc_outline = ski.segmentation.find_boundaries(cyto_mask,mode="outer")
nuc_outline = np.ma.masked_where(nuc_outline==False,nuc_outline)

fig,ax = plt.subplots(figsize=(20,20))
ax.imshow(ski.exposure.rescale_intensity(gfp[15000:20000,25000:30000], in_range=(0,3000)))
#show = microshow(images=composite, cmaps=['pure_green','pure_magenta'], limits=[[0,4000],[0,3500]], ax=ax)
ax.imshow(nuc_outline, cmap="Grays_r")

ax.set_yticks([])
ax.set_xticks([])
plt.tight_layout()

In [ ]:
tifffile.imwrite(nuclei_save_path, mask_nuclei.astype(img_dtype), dtype=img_dtype)

In [3]:
nuclei, cells, nuclei_per_cells = reconcile_nuclei_cells(nuclei_mask,cyto_mask)
tifffile.imwrite(cyto_save_path, cells.astype(img_dtype))
tifffile.imwrite(nuclei_save_path, nuclei.astype(img_dtype))


Nuclei per cell statistics:
--------------------------
Cells with 0 nuclei: 357
Cells with 1 nuclei: 3571
Cells with 2 nuclei: 124
Cells with 3 nuclei: 12
Cells with 4 nuclei: 2
--------------------------



In [ ]:
np.max(cells)

In [ ]:
nuclei, cells = tiled_reconcile_nuclei_cells(nuclei_tiles,cyto_tiles)
reconcile_nuclei = stitch.stitch_masks(nuclei)
reconcile_cells = stitch.stitch_masks(cells)
tifffile.imwrite(cyto_save_path, reconcile_cells.astype(img_dtype))
tifffile.imwrite(nuclei_save_path, reconcile_nuclei.astype(img_dtype))

In [ ]:
reconcile_cells.max()

In [ ]:
nuclei, cells = reconcile_nuclei_cells(nuclei_mask, cyto_mask, how='contained_in_cells')
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif", cells.astype(img_dtype))
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_nuclei_mask.tif", nuclei.astype(img_dtype))

In [ ]:
hasattr(cyto_tiles[0,0], "compute")

In [ ]:
# fig, ax = plt.subplots(ncols=2,figsize=(20,10))
fig, ax = plt.subplots(figsize=(10,10))

# ax[0].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(img[0], in_range=(0,4000)), nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)))
# ax[1].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(img_log[1], nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)), cmap='gray')


# # # ax[1].imshow(ski.color.label2rgb(mask))
# # # ax[1].set_yticks([])
# # # ax[1].set_xticks([])
# # # plt.tight_layout()


# # # # #img2 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]

# # # cy5 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]
# # # ax.imshow(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(cy5, in_range=(0,4000)), mask, color=(1,0,1), mode='thick'), cmap='gray')
ax.imshow(masks_cyto[9,17], cmap='gray')


# ax[0].imshow(img_tiles[9,7][0], cmap='gray')
# ax[1].imshow(img_tiles[9,7][1], cmap='gray')


ax.set_yticks([])
ax.set_xticks([])
# ax[0].set_yticks([])
# ax[0].set_xticks([])
# ax[1].set_yticks([])
# ax[1].set_xticks([])

plt.tight_layout()